Trong phần này sẽ có tiền xử lý + smote + gan + train test

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/DATN/fetal_health.csv'
data = pd.read_csv(file_path)
display(data.head())

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2126 entries, 0 to 2125
Data columns (total 22 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   baseline value                                          2126 non-null   float64
 1   accelerations                                           2126 non-null   float64
 2   fetal_movement                                          2126 non-null   float64
 3   uterine_contractions                                    2126 non-null   float64
 4   light_decelerations                                     2126 non-null   float64
 5   severe_decelerations                                    2126 non-null   float64
 6   prolongued_decelerations                                2126 non-null   float64
 7   abnormal_short_term_variability                         2126 non-null   float64
 8   mean_value_of_short_term_variability  

In [ ]:
data.duplicated().sum()


np.int64(13)

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.duplicated().sum()

np.int64(0)

In [ ]:
data.isna().sum().sum()

np.int64(0)

In [ ]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
baseline value,2113.0,133.304780,9.837451,106.0,126.000,133.000,140.000,160.000
accelerations,2113.0,0.003188,0.003871,0.0,0.000,0.002,0.006,0.019
fetal_movement,2113.0,0.009517,0.046804,0.0,0.000,0.000,0.003,0.481
uterine_contractions,2113.0,0.004387,0.002941,0.0,0.002,0.005,0.007,0.015
light_decelerations,2113.0,0.001901,0.002966,0.0,0.000,0.000,0.003,0.015
severe_decelerations,2113.0,0.000003,0.000057,0.0,0.000,0.000,0.000,0.001
prolongued_decelerations,2113.0,0.000159,0.000592,0.0,0.000,0.000,0.000,0.005
abnormal_short_term_variability,2113.0,46.993848,17.177782,12.0,32.000,49.000,61.000,87.000
mean_value_of_short_term_variability,2113.0,1.335021,0.884368,0.2,0.700,1.200,1.700,7.000
percentage_of_time_with_abnormal_long_term_variability,2113.0,9.795078,18.337073,0.0,0.000,0.000,11.000,91.000


In [ ]:
negative_values_found = False
for column in data.select_dtypes(include=['number']).columns:
    negative_count = (data[column] < 0).sum()
    if negative_count > 0:
        print(f"Cột '{column}' có {negative_count} giá trị nhỏ hơn 0.")
        negative_values_found = True

if not negative_values_found:
    print("Không tìm thấy giá trị nào nhỏ hơn 0 trong các cột số.")

Cột 'histogram_tendency' có 165 giá trị nhỏ hơn 0.


In [ ]:
data["histogram_tendency"].value_counts()

,count
histogram_tendency,
0.0,1110
1.0,838
-1.0,165


In [ ]:
data["fetal_health"].value_counts()

,count
fetal_health,
1.0,1646
2.0,292
3.0,175


Smote

In [ ]:
from imblearn.over_sampling import SMOTE
target_col = 'fetal_health'

print("Kích thước data gốc:", data.shape)
display(data.head())

# =============================
# 2. Tách X, y từ toàn bộ data
# =============================
X = data.drop(columns=[target_col]).copy()
y = data[target_col].copy().astype(int)

print("\nPhân bố lớp ban đầu:")
print(y.value_counts().sort_index())

# =============================
# 3. SMOTE trên toàn bộ data
# =============================
smote = SMOTE(
    sampling_strategy='auto',
    random_state=42,
    k_neighbors=5
)

X_smote, y_smote = smote.fit_resample(X, y)

X_smote = pd.DataFrame(X_smote, columns=X.columns)
y_smote = pd.Series(y_smote, name=target_col)

print("\nPhân bố lớp sau SMOTE toàn bộ data:")
print(y_smote.value_counts().sort_index())

print("\nKích thước sau SMOTE:")
print("X_smote:", X_smote.shape)
print("y_smote:", y_smote.shape)

Kích thước data gốc: (2113, 22)


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0



Phân bố lớp ban đầu:
fetal_health
1    1646
2     292
3     175
Name: count, dtype: int64

Phân bố lớp sau SMOTE toàn bộ data:
fetal_health
1    1646
2    1646
3    1646
Name: count, dtype: int64

Kích thước sau SMOTE:
X_smote: (4938, 21)
y_smote: (4938,)


In [ ]:
# Gộp X và y thành 1 DataFrame
data_smote = pd.concat([X_smote.reset_index(drop=True),
                        y_smote.reset_index(drop=True)], axis=1)

In [ ]:
data_smote

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,73.000000,0.500000,43.000000,...,62.000000,126.000000,2.000000,0.000000,120.000000,137.000000,121.000000,73.000000,1.000000,2
1,132.000000,0.006000,0.000000,0.006000,0.003000,0.0,0.000000,17.000000,2.100000,0.000000,...,68.000000,198.000000,6.000000,1.000000,141.000000,136.000000,140.000000,12.000000,0.000000,1
2,133.000000,0.003000,0.000000,0.008000,0.003000,0.0,0.000000,16.000000,2.100000,0.000000,...,68.000000,198.000000,5.000000,1.000000,141.000000,135.000000,138.000000,13.000000,0.000000,1
3,134.000000,0.003000,0.000000,0.008000,0.003000,0.0,0.000000,16.000000,2.400000,0.000000,...,53.000000,170.000000,11.000000,0.000000,137.000000,134.000000,137.000000,13.000000,1.000000,1
4,132.000000,0.007000,0.000000,0.008000,0.000000,0.0,0.000000,16.000000,2.400000,0.000000,...,53.000000,170.000000,9.000000,0.000000,137.000000,136.000000,138.000000,11.000000,1.000000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4933,132.993688,0.001002,0.000000,0.009006,0.003021,0.0,0.002994,61.000000,2.598738,0.000000,...,51.995792,183.985273,6.995792,0.000000,124.991585,96.004208,102.027350,107.970546,0.000000,3
4934,125.740577,0.000000,0.001435,0.006153,0.006130,0.0,0.001718,65.152716,2.502300,0.000000,...,63.000000,191.717572,6.259423,0.564856,105.564856,90.129712,110.305433,25.236418,-0.282428,3
4935,134.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,80.170412,0.200000,90.170412,...,133.000000,136.000000,1.000000,0.000000,135.000000,134.000000,136.000000,0.000000,1.000000,3
4936,134.918883,0.000000,0.012473,0.001243,0.000000,0.0,0.000000,68.797206,0.304056,78.121676,...,131.756648,140.878324,1.000000,0.000000,135.878324,135.837765,136.878324,0.000000,0.000000,3


In [ ]:
!pip install ctgan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 16.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from ctgan import CTGAN
from sklearn.model_selection import train_test_split

target_col = 'fetal_health'

print("Kích thước data gốc:", data_smote.shape)
display(data_smote.head())

print("\nPhân bố lớp ban đầu:")
print(data_smote[target_col].value_counts().sort_index())

# =============================
# 2. Train CTGAN trên toàn bộ data
#    fetal_health vẫn giữ dạng số
# =============================
gan_df = data_smote.copy()

ctgan = CTGAN(
    epochs=300,
    batch_size=100,
    verbose=True
)

ctgan.fit(gan_df, discrete_columns=[target_col])

print("\nĐã train xong CTGAN")

# =============================
# 3. Sinh thêm dữ liệu để cân bằng
# =============================
class_counts = gan_df[target_col].value_counts()
max_count = class_counts.max()

synthetic_parts = []

for cls, count in class_counts.items():
    need = max_count - count

    if need <= 0:
        continue

    print(f"\nLớp {cls} cần sinh thêm {need} mẫu")

    collected = []
    total_collected = 0

    while total_collected < need:
        sample_n = max(500, need * 2)
        fake_batch = ctgan.sample(sample_n)

        # vì target là số, lọc trực tiếp theo số
        fake_cls = fake_batch[fake_batch[target_col] == cls].copy()

        if len(fake_cls) > 0:
            collected.append(fake_cls)
            total_collected += len(fake_cls)
            print(f"Đã lấy được {total_collected}/{need}")

    fake_cls_final = pd.concat(collected, axis=0).iloc[:need].copy()
    synthetic_parts.append(fake_cls_final)

# =============================
# 4. Gộp dữ liệu thật + synthetic
# =============================
if len(synthetic_parts) > 0:
    synthetic_df = pd.concat(synthetic_parts, axis=0).reset_index(drop=True)
    balanced_df = pd.concat([gan_df, synthetic_df], axis=0).reset_index(drop=True)
else:
    balanced_df = gan_df.copy()

print("\nPhân bố lớp sau CTGAN:")
print(balanced_df[target_col].value_counts().sort_index())

# =============================
# 5. Tách X, y
# =============================
X = balanced_df.drop(columns=[target_col]).copy()
y = balanced_df[target_col].astype(int)

# =============================
# 6. Chia train/test
# =============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nKích thước tập train/test:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nPhân bố y_train:")
print(y_train.value_counts().sort_index())

print("\nPhân bố y_test:")
print(y_test.value_counts().sort_index())

Kích thước data gốc: (4938, 22)


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1



Phân bố lớp ban đầu:
fetal_health
1    1646
2    1646
3    1646
Name: count, dtype: int64


Gen. (-01.88) | Discrim. (+00.13):  10%|█         | 30/300 [01:31<13:45,  3.06s/it]


KeyboardInterrupt: 

In [ ]:
import numpy as np
y = np.array(y).astype(int)

In [ ]:
from imblearn.over_sampling import SMOTE

# khởi tạo SMOTE
smote = SMOTE(random_state=42)

# áp dụng vào tập train
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Khởi tạo model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# Train model với dữ liệu đã scale
rf_model.fit(X_train_scaled, y_train)

# Predict
y_pred = rf_model.predict(X_test_scaled)

# Đánh giá
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9527186761229315

Classification Report:
               precision    recall  f1-score   support

         1.0       0.95      1.00      0.97       330
         2.0       0.93      0.72      0.82        58
         3.0       0.97      0.91      0.94        35

    accuracy                           0.95       423
   macro avg       0.95      0.88      0.91       423
weighted avg       0.95      0.95      0.95       423


Confusion Matrix:
 [[329   1   0]
 [ 15  42   1]
 [  1   2  32]]
